#  Lab: Vision Transformers + Explainability (PyTorch)

By Nicolas Echevarrieta-Catalan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/schwartz-cnl/Computational-Neuroscience-Class/blob/main/Transformers%20and%20Self-Attention/Transformers_and_CNN_explainability.ipynb
)

 **Learning goals**
 1) Instantiate ViT models from HuggingFace + timm
 2) Run inference on images
 3) Visualize attention maps for different heads/layers
 4) Apply Grad-CAM and Grad-CAM++ to ViTs and CNNs
 5) Compare explanations across architectures and weight sources


 **Vision Transformer (ViT)** is an architecture based on Google's paper [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929), which is the adaptation to image classification fo the now omnipresent Transformer architecture (chat-gpt, geminy, etc... are Transformers) presented in, also Google's, paper [Attention is all you need](https://arxiv.org/abs/1706.03762)

 In this lab, we will use this architecture to predict the class of some images, and extract the models **raw attention scores** as an explainability method. Latter, we will compare this explanations with more "clasical" methods based on gradients on ViTs and the previous main architeture for image classification, Convolutional Neural Networks (CNN).



## 0) Setup (install deps)
Run this cell once in your environment.

In [ ]:
# If running in Colab, uncomment and run ONCE per colab session:
!pip install -q timm transformers grad-cam opencv-python



## 1) Imports

In [ ]:
import os
import math
import requests
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from PIL import Image

import torchvision
from torchvision import transforms

import timm
from timm.data import resolve_model_data_config, create_transform
from transformers import ViTForImageClassification, ViTImageProcessor

# pytorch-grad-cam
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


## 2) Constants / Configuration

In [ ]:
@dataclass
class Config:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 0

    # Images: local paths or URLs
    image_sources: Tuple[str, ...] = (
        # You can replace these with local paths.
        # If URLs are blocked in your setup, use local files instead.
        "https://images.unsplash.com/photo-1518791841217-8f162f1e1131",  # Cat
        "https://images.unsplash.com/photo-1546182990-dffeafbe841d",      # A cat but bigger than a lot of dogs (some dogs are bigger)
        "https://images.rawpixel.com/image_800/cHJpdmF0ZS9sci9pbWFnZXMvd2Vic2l0ZS8yMDIyLTA1L25zODIzMC1pbWFnZS5qcGc.jpg", # Dog
    )

    # HuggingFace ViT
    hf_model_name: str = "google/vit-base-patch16-224"

    # timm ViT (same architecture family; compare weight sources)
    timm_model_name: str = "vit_base_patch16_224"                  # ImageNet-1k pretrained

    # CNNs
    cnn_resnet: str = "resnet50"
    cnn_vgg: str = "vgg16"

cfg = Config()

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

print("Device:", cfg.device)

## 3) Helper functions (I/O, plotting, preprocessing, utilities)

In [ ]:
def load_image_from_source(source: str, resize_if_needed: Optional[int] = None) -> Image.Image:
    """
    Load an image from a local file path or a URL.
    Returns a PIL image in RGB.
    """
    if source.startswith("http://") or source.startswith("https://"):
        r = requests.get(source, stream=True, timeout=20)
        r.raise_for_status()
        img = Image.open(r.raw).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")

    if resize_if_needed is not None:
        img = img.resize((resize_if_needed, resize_if_needed))
    return img

def show_images(images: List[Image.Image], titles: Optional[List[str]] = None, cols: int = 2, figsize=(10, 6)):
    rows = math.ceil(len(images) / cols)
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        if titles is not None:
            plt.title(titles[i], fontsize=10)
    plt.tight_layout()
    plt.show()

def topk_probs(logits: torch.Tensor, k: int = 5) -> Tuple[torch.Tensor, torch.Tensor]:
    probs = F.softmax(logits, dim=-1)
    vals, idxs = torch.topk(probs, k=k, dim=-1)
    return vals, idxs

def denorm_for_display(img_t: torch.Tensor, mean: List[float], std: List[float]) -> np.ndarray:
    """
    img_t: (3,H,W) tensor normalized by mean/std.
    returns float RGB in [0,1] for visualization.
    """
    img = img_t.detach().cpu().clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

def plot_attention_grid(attn_map: np.ndarray, title: str = "", cmap: str = "viridis"):
    """
    attn_map: (grid_h, grid_w) numpy
    """
    plt.figure(figsize=(4, 4))
    plt.imshow(attn_map, cmap=cmap)
    plt.axis("off")
    plt.title(title, fontsize=10)
    plt.show()

def patch_grid_size(image_size: int, patch_size: int) -> int:
    assert image_size % patch_size == 0
    return image_size // patch_size


IMAGENET_1K_LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"

def load_imagenet_labels() -> List[str]:
    r = requests.get(IMAGENET_1K_LABELS_URL, timeout=20)
    r.raise_for_status()
    return r.text.strip().splitlines()

def print_topk(top_vals, top_idxs, model_name: str):
    print("\n====", model_name, "====")
    for i in range(top_idxs.shape[0]):
        print(f"\nImage {i}:")
        for j in range(5):
            cls_id = int(top_idxs[i, j])
            prob = float(top_vals[i, j])
            if imagenet_labels is None:
                label = str(cls_id)
            else:
                label = imagenet_labels[cls_id]
            print(f"  {j+1:>2d}. {label:>25s}   p={prob:.3f}")

## 4) Load images (you can change them)

In [ ]:
# To include more images, just add their URL to Config.image_sources
# Try to load labels (if URL blocked, fall back to class index)
try:
    imagenet_labels = load_imagenet_labels()
    print("Loaded ImageNet labels:", len(imagenet_labels))
    print("\n\nYou can use imagef of any of this categories\n", load_imagenet_labels())
except Exception as e:
    imagenet_labels = None
    print("Could not load ImageNet labels; will print class indices only.\nError:", e)

In [ ]:
images = [load_image_from_source(s) for s in cfg.image_sources]
show_images(images, titles=[f"Image {i}" for i in range(len(images))], cols=2)

# Part 1 — HuggingFace ViT: prediction + attention visualization

Here we will instantiate a ViT (Vision Transformer) model with pretrained weights from a library called HuggingFace. This library is currently one of the most used sources for ML models.

HF weight for ViT come from the original paper: [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)

## 1.0) Helper functions

In [ ]:
@torch.no_grad()
def hf_predict(images: List[Image.Image], hf_processor, hf_model) -> Dict:
    inputs = hf_processor(images=images, return_tensors="pt").to(cfg.device)
    outputs = hf_model(**inputs)
    logits = outputs.logits
    top_vals, top_idxs = topk_probs(logits, k=5)
    return {
        "inputs": inputs,
        "outputs": outputs,
        "top_vals": top_vals.detach().cpu(),
        "top_idxs": top_idxs.detach().cpu(),
    }

@torch.no_grad()
def hf_get_attentions(images: List[Image.Image], hf_processor, hf_model) -> List[torch.Tensor]:
    inputs = hf_processor(images=images, return_tensors="pt").to(cfg.device)
    outputs = hf_model(**inputs, output_attentions=True)
    # outputs.attentions is a tuple of length num_layers
    # each element: (B, heads, tokens, tokens)
    return list(outputs.attentions), inputs

def hf_cls_to_patch_attention(attn_layer: torch.Tensor, image_index: int, head_index: int) -> np.ndarray:
    """
    Extract CLS->patch attention for a single image/head from a single layer.
    Returns (grid_h, grid_w) numpy.
    """
    # attn_layer: (B, heads, tokens, tokens)
    attn = attn_layer[image_index, head_index]          # (tokens, tokens)
    cls_to_all = attn[0, 1:]                            # CLS token to patch tokens
    num_patches = cls_to_all.shape[0]

    # For ViT-base patch16 @ 224: 14x14 = 196 patches
    grid = int(math.sqrt(num_patches))
    assert grid * grid == num_patches, "Num patches not a perfect square; check image/patch size."

    attn_map = cls_to_all.reshape(grid, grid).detach().cpu().numpy()
    return attn_map

## 1.1) Load HF ViT + processor

Here we instantia an instance of the preprocesing steps and the model, configuring it so that attention scores can be extracted

In [ ]:
hf_processor = ViTImageProcessor.from_pretrained(cfg.hf_model_name)
hf_model = ViTForImageClassification.from_pretrained(
    cfg.hf_model_name,
    attn_implementation="eager",
).to(cfg.device).eval()

# Make HF return attentions
hf_model.config.output_attentions = True
hf_model.config.return_dict = True  # ensures outputs is a dict-like object

print("HF model loaded:", cfg.hf_model_name)

## 1.2) Inference (HF) + top-k predictions
Using the instantiated model, we can predict the class for each image. HuggingFace provides `id2label` for human-readable class names.

In [ ]:
hf_out = hf_predict(images, hf_processor, hf_model)

for i in range(len(images)):
    print(f"\nImage {i}:")
    for j in range(5):
        cls_id = int(hf_out["top_idxs"][i, j])
        prob = float(hf_out["top_vals"][i, j])
        label = hf_model.config.id2label.get(cls_id, str(cls_id))
        print(f"  {j+1:>2d}. {label:>25s}   p={prob:.3f}")

## 1.3) Attention maps (HF)
We’ll run a forward pass with `output_attentions=True`. This will extract the attention scores for all the loaded images.

If you add more images, you will have to re-run this cell to see their attention scores.

Attentions shape per layer: **(B, heads, tokens, tokens)**  
A common visualization: **CLS token attention to patch tokens** → reshape to patch grid.

In [ ]:
attentions_hf, hf_inputs_for_attn = hf_get_attentions(images, hf_processor, hf_model)
print("HF layers:", len(attentions_hf), "attn shape:", attentions_hf[0].shape)

In [ ]:
# Pick a layer/head to visualize
image_index = 0
layer_index = -1     # last layer
head_index = 0

attn_map = hf_cls_to_patch_attention(attentions_hf[layer_index], image_index=image_index, head_index=head_index)
plot_attention_grid(attn_map, title=f"HF ViT | layer {layer_index} head {head_index} | CLS→patch")

In [ ]:
# Visualize ALL heads from one layer (3x4) with heatmap overlay
image_index = 0
layer_index = -1  # values between [-12, 11]. {-1} and {11} are last layer

num_heads = attentions_hf[layer_index].shape[1]  # should be 12 for ViT-Base
out_size = 224  # or hf_image_size if you defined it

plt.figure(figsize=(10, 7))
for h in range(num_heads):
    plt.subplot(3, 4, h + 1)

    m = hf_cls_to_patch_attention(
        attentions_hf[layer_index],
        image_index=image_index,
        head_index=h
    )

    plt.imshow(images[image_index].resize((out_size, out_size)))
    plt.imshow(m, cmap="jet", alpha=0.5, extent=(0, out_size, out_size, 0))

    plt.axis("off")
    plt.title(f"head {h}", fontsize=10)

plt.suptitle(f"HF ViT attention overlay | layer {layer_index} | image {image_index}", fontsize=12)
plt.tight_layout()
plt.show()


Using the cat image (index 0) we can observe some head focusing in the cat (1, 3 and 8), some seem to be looking at the background (2, 5 and 10), while others seem to be looking at random places of the image.

Having a set of heads that look at semingly unimportant areas can happen as models are optimized for classification, not for diverse interpretable attention, and some heads become redundant. There are papers showing you can prune many heads with little performance drop.

**This situation is completly dependant of the set of weights you are using.**

PD: A famous paper on the topic of pruning applied to any artificial neural network is "The Lottery Ticket Hypothesis" https://arxiv.org/abs/1803.03635

**FOR STUDENTS**

Try changing the image and layer attention used to plot attention scores and see how the attention of the model changes

# Part 2 — timm ViT: prediction + attention visualization

Timm is another ML library for pytorch. The main difference is that the set of weights of their models are often train by themselves.


## 2.0) Helper functions

In [ ]:
@torch.no_grad()
def timm_predict(vit: torch.nn.Module, tfm, images: List[Image.Image]):
    x = torch.stack([tfm(img) for img in images], dim=0).to(cfg.device)  # (B,3,H,W)
    logits = vit(x)
    top_vals, top_idxs = topk_probs(logits, k=5)
    return x, logits, top_vals.detach().cpu(), top_idxs.detach().cpu()


# We wrap each block's attention forward to store the attention matrix (after softmax).

def enable_timm_attention_capture(vit: torch.nn.Module):
    """
    Monkey-patch timm ViT blocks so each block.attn stores `last_attn` with shape:
      (B, heads, tokens, tokens)
    """
    for block in vit.blocks:
        attn_mod = block.attn
        if hasattr(attn_mod, "_original_forward"):
            continue  # already patched

        attn_mod._original_forward = attn_mod.forward

        def forward_with_capture(x, attn_mod=attn_mod, **kwargs):
            B, N, C = x.shape
            qkv = attn_mod.qkv(x).reshape(B, N, 3, attn_mod.num_heads, C // attn_mod.num_heads)
            qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, heads, N, head_dim)
            q, k, v = qkv[0], qkv[1], qkv[2]

            attn = (q @ k.transpose(-2, -1)) * attn_mod.scale
            attn = attn.softmax(dim=-1)
            attn_mod.last_attn = attn.detach()

            attn = attn_mod.attn_drop(attn)
            x = (attn @ v).transpose(1, 2).reshape(B, N, C)
            x = attn_mod.proj(x)
            x = attn_mod.proj_drop(x)
            return x

        attn_mod.forward = forward_with_capture


def get_timm_last_attentions(vit: torch.nn.Module) -> List[torch.Tensor]:
    """
    After a forward pass, returns a list of per-block attention matrices.
    """
    attns = []
    for block in vit.blocks:
        attn_mod = block.attn
        if not hasattr(attn_mod, "last_attn"):
            attns.append(None)
        else:
            attns.append(attn_mod.last_attn)
    return attns


def timm_cls_to_patch_attention(attn_layer: torch.Tensor, image_index: int, head_index: int) -> np.ndarray:
    """
    attn_layer: (B, heads, tokens, tokens) from timm patched capture.
    Returns (grid_h, grid_w) numpy.
    """
    attn = attn_layer[image_index, head_index]   # (tokens, tokens)
    cls_to_all = attn[0, 1:]                     # CLS->patch
    num_patches = cls_to_all.shape[0]
    grid = int(math.sqrt(num_patches))
    assert grid * grid == num_patches
    return cls_to_all.reshape(grid, grid).detach().cpu().numpy()


## 2.1)  Load timm ViT + processor

In [ ]:
# timm provides model-specific normalization config.
def timm_transform_for_model(model: torch.nn.Module):
    data_cfg = resolve_model_data_config(model)
    tfm = create_transform(**data_cfg, is_training=False)
    return tfm, data_cfg

timm_vit = timm.create_model(cfg.timm_model_name, pretrained=True).to(cfg.device).eval()
enable_timm_attention_capture(timm_vit)
timm_tfm, timm_cfg = timm_transform_for_model(timm_vit)
print("timm:", cfg.timm_model_name, timm_cfg["input_size"], timm_cfg["mean"], timm_cfg["std"])



## 2.2) Inference (timm) + top-k predictions

In [ ]:
x, logits, topv, topi = timm_predict(timm_vit, timm_tfm, images)
print_topk(topv, topi, cfg.timm_model_name)

## 2.3) Attention maps (timm)

In [ ]:
attentions_timm = get_timm_last_attentions(timm_vit)
print("Blocks:", len(attentions_timm), "example attn shape:", attentions_timm[-1].shape)

In [ ]:
image_index = 0
block_index = -1
head_index = 0

m_timm = timm_cls_to_patch_attention(attentions_timm[block_index], image_index=image_index, head_index=head_index)
plot_attention_grid(m_timm, title=f"timm A | block {block_index} head {head_index} | CLS→patch")


In [ ]:
image_index = 0
block_index = -1  # values between [-12, 11]. {-1} and {11} are last layer

num_heads = attentions_timm[block_index].shape[1]  # should be 12 for ViT-Base
out_size = 224  # or timm_image_size if you defined it

plt.figure(figsize=(10, 7))
for h in range(num_heads):
    plt.subplot(3, 4, h + 1)

    m = timm_cls_to_patch_attention(attentions_timm[block_index], image_index, h)

    # overlay
    plt.imshow(images[image_index].resize((out_size, out_size)))
    plt.imshow(m, cmap="jet", alpha=0.5, extent=(0, out_size, out_size, 0))

    plt.axis("off")
    plt.title(f"head {h}", fontsize=10)

plt.suptitle(f"timm attention overlay | block {block_index} | image {image_index}", fontsize=12)
plt.tight_layout()
plt.show()


As said before, the attention scores are completly dependant on the set of weights used, even if the architecture is the same. Here last layer for image 0 (cat) is mainly useless for human interpretation.

**FOR STUDENTS**

Try changing the image and layer attention used to plot attention scores and see how the attention of the model changes. How is it similar or diferent to HuggingFace set of weights?

# Part 3 — ViT attention comparative
Let's compare ViT attentions for both models


In [ ]:
# HF vs timm attention heads side-by-side (4 rows x 3 groups)
image_index = 0
layer_index = -1
out_size = 224

fig, axes = plt.subplots(4, 6, figsize=(9, 6))
fig.suptitle(
    f"HF vs timm attention | image {image_index} | HF layer {layer_index} vs timm block {layer_index}",
    fontsize=14
)

for head in range(12):
    r = head // 3
    c_group = head % 3
    c_hf = 2 * c_group
    c_tm = 2 * c_group + 1

    m_hf = hf_cls_to_patch_attention(attentions_hf[layer_index], image_index, head)
    m_tm = timm_cls_to_patch_attention(attentions_timm[layer_index], image_index, head)

    # HF
    ax = axes[r, c_hf]
    ax.imshow(images[image_index].resize((out_size, out_size)))
    ax.imshow(m_hf, cmap="jet", alpha=0.5, extent=(0, out_size, out_size, 0))
    ax.set_title(f"h{head} HF", fontsize=9)
    ax.axis("off")

    # timm
    ax = axes[r, c_tm]
    ax.imshow(images[image_index].resize((out_size, out_size)))
    ax.imshow(m_tm, cmap="jet", alpha=0.5, extent=(0, out_size, out_size, 0))
    ax.set_title(f"h{head} timm", fontsize=9)
    ax.axis("off")

plt.tight_layout()

# ---- Add vertical separator lines ----
# Get x positions between column 1-2 and 3-4 (in figure coordinates)
col2_right = axes[0,1].get_position().x1
col3_left  = axes[0,2].get_position().x0
col4_right = axes[0,3].get_position().x1
col5_left  = axes[0,4].get_position().x0

x_sep1 = (col2_right + col3_left) / 2
x_sep2 = (col4_right + col5_left) / 2

# Draw vertical lines spanning full figure height
fig.add_artist(plt.Line2D([x_sep1, x_sep1], [0.05, 0.95], transform=fig.transFigure, color='black', linewidth=2))
fig.add_artist(plt.Line2D([x_sep2, x_sep2], [0.05, 0.95], transform=fig.transFigure, color='black', linewidth=2))
# ---------------------------------------

plt.show()

# Part 4 — Attention Rollout

As we saw, ttention at a single layer only shows local token interactions that are different for different heads. Attention rollout recursively multiplies attention matrices across layers to estimate how much each patch ultimately contributes to the classification token.

In [ ]:
image_index = 0
out_size = 224

# Ensure attentions are available (reuse previous helpers)
if "attentions_hf" not in globals():
    attentions_hf, _ = hf_get_attentions(images, hf_processor, hf_model)

if "attentions_timm" not in globals():
    _x_timm, _logits_timm, *_ = timm_predict(timm_vit, timm_tfm, images)
    attentions_timm = get_timm_last_attentions(timm_vit)

img_pil = images[image_index].convert("RGB").resize((out_size, out_size))
rgb_float = np.asarray(img_pil).astype(np.float32) / 255.0


def attention_rollout(attentions, image_index=0):
    joint = None
    for A in attentions:
        A = A[image_index]          # (H, T, T)
        A = A.mean(dim=0)           # fuse heads -> (T, T)

        T = A.shape[-1]
        A = A + torch.eye(T, device=A.device)  # residual
        A = A / (A.sum(dim=-1, keepdim=True) + 1e-8)

        joint = A if joint is None else (A @ joint)

    cls_to_patch = joint[0, 1:]
    n = cls_to_patch.shape[0]
    g = int(math.sqrt(n))
    m = cls_to_patch.reshape(g, g)

    m = m - m.min()
    m = m / (m.max() + 1e-8)
    return m.detach().cpu().numpy()


def overlay_map(rgb_float, attn_map, smooth=False):
    attn = torch.tensor(attn_map).unsqueeze(0).unsqueeze(0)  # (1,1,g,g)

    if smooth:
        attn = torch.nn.functional.interpolate(
            attn, size=(out_size, out_size), mode="bilinear", align_corners=False
        )
    else:
        attn = torch.nn.functional.interpolate(
            attn, size=(out_size, out_size), mode="nearest"
        )

    attn = attn.squeeze().numpy()
    return show_cam_on_image(rgb_float, attn, use_rgb=True)


# Compute rollout maps
hf_roll = attention_rollout(attentions_hf, image_index=image_index)
timm_roll = attention_rollout(attentions_timm, image_index=image_index)

# Build overlays
hf_overlay = overlay_map(rgb_float, hf_roll, smooth=False)
hf_overlay_s = overlay_map(rgb_float, hf_roll, smooth=True)

timm_overlay = overlay_map(rgb_float, timm_roll, smooth=False)
timm_overlay_s = overlay_map(rgb_float, timm_roll, smooth=True)

# Plot: image | overlay | smoothed overlay
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
fig.suptitle(f"Attention Rollout | image {image_index}", fontsize=14)

# HF
axes[0, 0].imshow(rgb_float)
axes[0, 0].set_title("Original")
axes[0, 0].axis("off")

axes[0, 1].imshow(hf_overlay)
axes[0, 1].set_title("HF rollout overlay")
axes[0, 1].axis("off")

axes[0, 2].imshow(hf_overlay_s)
axes[0, 2].set_title("HF rollout smoothed")
axes[0, 2].axis("off")

# timm
axes[1, 0].imshow(rgb_float)
axes[1, 0].set_title("Original")
axes[1, 0].axis("off")

axes[1, 1].imshow(timm_overlay)
axes[1, 1].set_title("timm rollout overlay")
axes[1, 1].axis("off")

axes[1, 2].imshow(timm_overlay_s)
axes[1, 2].set_title("timm rollout smoothed")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

# Part 5 — Grad-CAM and Grad-CAM++ on ViTs

In thnis part we will use gradient based method to obtain exlpanations. Even though this methods were designed to CNNs, there is an adaptation to transformers in `pytorch-grad-cam` library.

**Class Activation Map (CAM)** methods generate their explanations going from the class prediction to the input image through the model using the gradients. What this means is that **the explanation is specific for a given prediction**. You can "ask" the model why it thinks that there is a cat in the model, or why it thinks that there is an airplane (even if there is no airplane). Different methods obtain the explanations with different mathematical procedures, and often give very different results.

The two selected methods are **Grad-CAM**, one of the first methods, and **Grad-CAM++**, which main difference with Grad-CAM is that it uses second order gradients instead of first order (no need to understand what it means, you can think it uses magic in a different way).


## 4.0) Helper functions

In [ ]:
def vit_reshape_transform(tensor: torch.Tensor) -> torch.Tensor:
    """
    CAM needs (B,C,H,W). ViT tokens come as (B, tokens, C) for common internal layers.
    We drop CLS (token 0), reshape patches to a square grid, and permute to (B,C,H,W).
    """
    # tensor: (B, tokens, C)
    tensor = tensor[:, 1:, :]  # drop CLS -> (B, num_patches, C)
    b, n, c = tensor.shape
    h = w = int(math.sqrt(n))
    assert h * w == n, f"num_patches={n} not a perfect square (check input size / patch size)."
    return tensor.reshape(b, h, w, c).permute(0, 3, 1, 2).contiguous()

## 4.1) Grad-CAM for HuggingFace ViT

In [ ]:
image_index = 0
out_size = 224

# For CAM overlay we need the *original* image as float RGB in [0,1]
img_pil = images[image_index].convert("RGB").resize((out_size, out_size))
rgb_float = np.asarray(img_pil).astype(np.float32) / 255.0

In [ ]:
# ==========================
# HF ViT CAMs (robust: force tensor logits output)
# ==========================

_hf_base = hf_model.model if (hasattr(hf_model, "model") and not isinstance(hf_model, torch.nn.Module)) else hf_model

class HFLogitsWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

    def forward(self, pixel_values):
        out = self.base(pixel_values=pixel_values)
        # HF image classification models usually return an object/dict with .logits
        if hasattr(out, "logits"):
            return out.logits
        if isinstance(out, dict) and "logits" in out:
            return out["logits"]
        # fallback: if it's already a tensor
        return out

hf_cam_model = HFLogitsWrapper(_hf_base).to(cfg.device)
hf_cam_model.eval()

hf_inputs = hf_processor(images=[images[image_index]], return_tensors="pt").to(cfg.device)

with torch.no_grad():
    hf_logits = hf_cam_model(hf_inputs["pixel_values"])
hf_pred = int(hf_logits.argmax(dim=1).item())
hf_label = _hf_base.config.id2label.get(hf_pred, str(hf_pred))

# target layer selection
_last_hf_block = _hf_base.vit.encoder.layer[-1]

hf_target_layers = [_last_hf_block.layernorm_before]

hf_targets = [ClassifierOutputTarget(hf_pred)]

hf_cam =  GradCAM(
    model=hf_cam_model,
    target_layers=hf_target_layers,
    reshape_transform=vit_reshape_transform,
)
hf_cam_vis = show_cam_on_image(rgb_float, hf_cam(input_tensor=hf_inputs["pixel_values"], targets=hf_targets)[0], use_rgb=True)

hf_campp = GradCAMPlusPlus(
    model=hf_cam_model,
    target_layers=hf_target_layers,
    reshape_transform=vit_reshape_transform,
)
hf_campp_vis = show_cam_on_image(rgb_float, hf_campp(input_tensor=hf_inputs["pixel_values"], targets=hf_targets)[0], use_rgb=True)

## 4.2) Grad-CAM for timm ViT

In [ ]:
timm_x = torch.stack([timm_tfm(images[image_index])], dim=0).to(cfg.device)
with torch.no_grad():
    timm_logits = timm_vit(timm_x)
timm_pred = int(timm_logits.argmax(dim=1).item())

timm_target_layers = [timm_vit.blocks[-1].norm1]
timm_targets = [ClassifierOutputTarget(timm_pred)]

timm_cam = GradCAM(
    model=timm_vit,
    target_layers=timm_target_layers,
    reshape_transform=vit_reshape_transform,
)
timm_cam_vis = show_cam_on_image(rgb_float, timm_cam(input_tensor=timm_x, targets=timm_targets)[0], use_rgb=True)

timm_campp =  GradCAMPlusPlus(
    model=timm_vit,
    target_layers=timm_target_layers,
    reshape_transform=vit_reshape_transform,
)
timm_campp_vis = show_cam_on_image(rgb_float, timm_campp(input_tensor=timm_x, targets=timm_targets)[0], use_rgb=True)

## 4.3) Grad-CAM comparison for ViT

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
fig.suptitle(f"Grad-CAM vs Grad-CAM++ | ViT | image {image_index}", fontsize=14)

# HF row
axes[0, 0].imshow(rgb_float)
axes[0, 0].set_title(f"Original (HF)\n{hf_label}")
axes[0, 0].axis("off")

axes[0, 1].imshow(hf_cam_vis)
axes[0, 1].set_title("HF Grad-CAM")
axes[0, 1].axis("off")

axes[0, 2].imshow(hf_campp_vis)
axes[0, 2].set_title("HF Grad-CAM++")
axes[0, 2].axis("off")

# timm row
axes[1, 0].imshow(rgb_float)
axes[1, 0].set_title(f"Original (timm)\nclass id {timm_pred}")
axes[1, 0].axis("off")

axes[1, 1].imshow(timm_cam_vis)
axes[1, 1].set_title("timm Grad-CAM")
axes[1, 1].axis("off")

axes[1, 2].imshow(timm_campp_vis)
axes[1, 2].set_title("timm Grad-CAM++")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

# Part 6 — Grad-CAM and Grad-CAM++ on CNNs

In this section we are going to use the same CAM methods, but on the architectures they were originaly designed for. We will use 2 CNN of great relevance:

- **VGG** was a very popular model with a modualr design where all the other models at that time were designed specifically. [Paper](https://arxiv.org/abs/1409.1556)

- **ResNet** is an improvement over VGG which introduced the concept of Residual Connection, allowing the information of the input to be propagated through all the network unalterated, instead of only it's transformation for the sucesive layers. This residual connection is a component present in almost any moder deep learning architecture. [Paper](https://arxiv.org/abs/1512.03385)

In [ ]:
image_index = 0
out_size = 224

# Original image (for overlay)
img_pil = images[image_index].convert("RGB").resize((out_size, out_size))
rgb_float = np.asarray(img_pil).astype(np.float32) / 255.0

# CNN input tensor (reuse the same ImageNet-normalized transform used for timm)
x_cnn = torch.stack([timm_tfm(images[image_index])], dim=0).to(cfg.device)

def _cnn_logits(model, x):
    out = model(x)
    if hasattr(out, "logits"):
        return out.logits
    if isinstance(out, dict) and "logits" in out:
        return out["logits"]
    if isinstance(out, (tuple, list)):
        return out[0]
    return out

# ==========================
# 5.0) Instantiate models
# ==========================

vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT).to(cfg.device).eval()
resnet18 = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT).to(cfg.device).eval()

has_labels = "imagenet_labels" in globals()

# ==========================
# 5.1) VGG CAMs
# ==========================
with torch.no_grad():
    vgg_logits = _cnn_logits(vgg, x_cnn)
vgg_pred = int(vgg_logits.argmax(dim=1).item())
vgg_targets = [ClassifierOutputTarget(vgg_pred)]
vgg_label = imagenet_labels[vgg_pred] if has_labels else f"class id {vgg_pred}"

# Target layer: last conv layer in VGG features (last module is ReLU, so pick last Conv2d)
_vgg_convs = [m for m in vgg.features.modules() if isinstance(m, torch.nn.Conv2d)]
if len(_vgg_convs) == 0:
    raise RuntimeError("Could not find a Conv2d layer inside vgg.features for CAM.")
vgg_target_layers = [_vgg_convs[-1]]

vgg_cam = GradCAM(model=vgg, target_layers=vgg_target_layers)
vgg_cam_vis = show_cam_on_image(rgb_float, vgg_cam(input_tensor=x_cnn, targets=vgg_targets)[0], use_rgb=True)

vgg_campp = GradCAMPlusPlus(model=vgg, target_layers=vgg_target_layers)
vgg_campp_vis = show_cam_on_image(rgb_float, vgg_campp(input_tensor=x_cnn, targets=vgg_targets)[0], use_rgb=True)

# ==========================
# 5.2) ResNet18 CAMs
# ==========================
with torch.no_grad():
    rn_logits = _cnn_logits(resnet18, x_cnn)
rn_pred = int(rn_logits.argmax(dim=1).item())
rn_targets = [ClassifierOutputTarget(rn_pred)]
rn_label = imagenet_labels[rn_pred] if has_labels else f"class id {rn_pred}"

# Target layer: ResNet last block output
rn_target_layers = [resnet18.layer4]

rn_cam = GradCAM(model=resnet18, target_layers=rn_target_layers)
rn_cam_vis = show_cam_on_image(rgb_float, rn_cam(input_tensor=x_cnn, targets=rn_targets)[0], use_rgb=True)

rn_campp = GradCAMPlusPlus(model=resnet18, target_layers=rn_target_layers)
rn_campp_vis = show_cam_on_image(rgb_float, rn_campp(input_tensor=x_cnn, targets=rn_targets)[0], use_rgb=True)

# ==========================
# 5.3) Plot
# ==========================
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
fig.suptitle(f"Grad-CAM vs Grad-CAM++ | CNNs | image {image_index}", fontsize=14)

# VGG row
axes[0, 0].imshow(rgb_float)
axes[0, 0].set_title(f"Original (VGG)\n{vgg_label}")
axes[0, 0].axis("off")

axes[0, 1].imshow(vgg_cam_vis)
axes[0, 1].set_title("VGG Grad-CAM")
axes[0, 1].axis("off")

axes[0, 2].imshow(vgg_campp_vis)
axes[0, 2].set_title("VGG Grad-CAM++")
axes[0, 2].axis("off")

# ResNet row
axes[1, 0].imshow(rgb_float)
axes[1, 0].set_title(f"Original (ResNet18)\n{rn_label}")
axes[1, 0].axis("off")

axes[1, 1].imshow(rn_cam_vis)
axes[1, 1].set_title("ResNet18 Grad-CAM")
axes[1, 1].axis("off")

axes[1, 2].imshow(rn_campp_vis)
axes[1, 2].set_title("ResNet18 Grad-CAM++")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

# Part 7 — Final comparison

In [ ]:

N = min(5, len(images))
out_size = 224

def _rgb_float(pil_img):
    pil_img = pil_img.convert("RGB").resize((out_size, out_size))
    return np.asarray(pil_img).astype(np.float32) / 255.0

def _overlay_from_gridmap(rgb_float, grid_map, smooth=True):
    attn = torch.tensor(grid_map).unsqueeze(0).unsqueeze(0)
    attn = torch.nn.functional.interpolate(
        attn,
        size=(out_size, out_size),
        mode="bilinear" if smooth else "nearest",
        align_corners=False if smooth else None,
    )
    attn = attn.squeeze().cpu().numpy()
    return show_cam_on_image(rgb_float, attn, use_rgb=True)

row_names = [
    "Original",
    "HF Rollout",
    "timm Rollout",
    "HF Grad-CAM",
    "timm Grad-CAM",
    "VGG Grad-CAM",
    "ResNet18 Grad-CAM",
    "HF Grad-CAM++",
    "timm Grad-CAM++",
    "VGG Grad-CAM++",
    "ResNet18 Grad-CAM++",
]
R = len(row_names)

fig, axes = plt.subplots(R, N, figsize=(3.0 * N, 2.0 * R))

if N == 1:
    axes = np.expand_dims(axes, axis=1)

for i in range(N):
    rgb = _rgb_float(images[i])

    # --- Rollouts ---
    hf_roll   = attention_rollout(attentions_hf, image_index=i)
    timm_roll = attention_rollout(attentions_timm, image_index=i)

    hf_roll_vis   = _overlay_from_gridmap(rgb, hf_roll, smooth=True)
    timm_roll_vis = _overlay_from_gridmap(rgb, timm_roll, smooth=True)

    # --- HF CAMs ---
    hf_inputs_i = hf_processor(images=[images[i]], return_tensors="pt").to(cfg.device)
    with torch.no_grad():
        hf_logits_i = hf_cam_model(hf_inputs_i["pixel_values"])
    hf_pred_i = int(hf_logits_i.argmax(dim=1).item())
    hf_targets_i = [ClassifierOutputTarget(hf_pred_i)]

    hf_cam_map   = hf_cam(input_tensor=hf_inputs_i["pixel_values"], targets=hf_targets_i)[0]
    hf_campp_map = hf_campp(input_tensor=hf_inputs_i["pixel_values"], targets=hf_targets_i)[0]

    hf_cam_vis   = show_cam_on_image(rgb, hf_cam_map, use_rgb=True)
    hf_campp_vis = show_cam_on_image(rgb, hf_campp_map, use_rgb=True)

    # --- timm CAMs ---
    x_timm_i = torch.stack([timm_tfm(images[i])], dim=0).to(cfg.device)
    with torch.no_grad():
        timm_logits_i = timm_vit(x_timm_i)
    timm_pred_i = int(timm_logits_i.argmax(dim=1).item())
    timm_targets_i = [ClassifierOutputTarget(timm_pred_i)]

    timm_cam_map   = timm_cam(input_tensor=x_timm_i, targets=timm_targets_i)[0]
    timm_campp_map = timm_campp(input_tensor=x_timm_i, targets=timm_targets_i)[0]

    timm_cam_vis   = show_cam_on_image(rgb, timm_cam_map, use_rgb=True)
    timm_campp_vis = show_cam_on_image(rgb, timm_campp_map, use_rgb=True)

    # --- CNN CAMs ---
    x_cnn_i = x_timm_i
    with torch.no_grad():
        vgg_logits_i = vgg(x_cnn_i)
        rn_logits_i  = resnet18(x_cnn_i)

    vgg_pred_i = int(vgg_logits_i.argmax(dim=1).item())
    rn_pred_i  = int(rn_logits_i.argmax(dim=1).item())

    vgg_targets_i = [ClassifierOutputTarget(vgg_pred_i)]
    rn_targets_i  = [ClassifierOutputTarget(rn_pred_i)]

    vgg_cam_map   = vgg_cam(input_tensor=x_cnn_i, targets=vgg_targets_i)[0]
    vgg_campp_map = vgg_campp(input_tensor=x_cnn_i, targets=vgg_targets_i)[0]
    rn_cam_map    = rn_cam(input_tensor=x_cnn_i, targets=rn_targets_i)[0]
    rn_campp_map  = rn_campp(input_tensor=x_cnn_i, targets=rn_targets_i)[0]

    vgg_cam_vis   = show_cam_on_image(rgb, vgg_cam_map, use_rgb=True)
    vgg_campp_vis = show_cam_on_image(rgb, vgg_campp_map, use_rgb=True)
    rn_cam_vis    = show_cam_on_image(rgb, rn_cam_map, use_rgb=True)
    rn_campp_vis  = show_cam_on_image(rgb, rn_campp_map, use_rgb=True)

    row_imgs = [
        rgb,
        hf_roll_vis,
        timm_roll_vis,
        hf_cam_vis,
        timm_cam_vis,
        vgg_cam_vis,
        rn_cam_vis,
        hf_campp_vis,
        timm_campp_vis,
        vgg_campp_vis,
        rn_campp_vis,
    ]

    for r in range(R):
        ax = axes[r, i]
        ax.imshow(row_imgs[r])
        ax.set_xticks([])
        ax.set_yticks([])

# --- Add clean Y-axis labels (only once per row) ---
for r in range(R):
    axes[r, 0].set_ylabel(
        row_names[r],
        fontsize=11,
        rotation=0,
        labelpad=60,
        va="center"
    )

# --- Column titles ---
for i in range(N):
    axes[0, i].set_title(f"Image {i}", fontsize=11)

plt.tight_layout()
plt.show()